# MCP: the tools live in a server

In sprint 3 the notebook owned the functions. Here the functions move into a small MCP server, and the notebook becomes an MCP client.

1. Start the server
2. The client and the server
3. What the server advertises
4. The loop
5. The same questions

The server is a separate process. The notebook only knows its URL. Any MCP-capable assistant can connect to the same URL.

# 1. Start the server

From the repo root, in a terminal:

```
python MLAppDev/sprint_3/seed.py
uvicorn MLAppDev.sprint_3.mcp.mcp_server:app --reload --port 8000
```

The first command writes `MLAppDev/sprint_3/data/app.db`. The second one serves the MCP server with uvicorn at `http://127.0.0.1:8000/mcp`. Keep that terminal open while you run this notebook.

The notebook does not start the server. That is the point: the server is its own process, and the URL is the contract.

# 2. The client and the server

The notebook imports `Client` from `mcp`, not the tool functions. `Client("http://...")` uses Streamable HTTP, so the tools are reached over the network instead of through an in-process function call.

In [1]:
import json
import os

from dotenv import load_dotenv
from mcp import Client
from openai import OpenAI

load_dotenv()
API_KEY = os.getenv("OPENROUTER_API_KEY")
if not API_KEY:
    from google.colab import userdata      # Colab Secrets
    API_KEY = userdata.get("OPENROUTER_API_KEY")

BASE_URL = "https://openrouter.ai/api/v1"
MODEL = "google/gemini-2.5-flash-lite"
MCP_SERVER_URL = "http://127.0.0.1:8000/mcp"

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

# 3. What the server advertises

`list_tools()` returns the tool names, descriptions and JSON Schemas. The server built them from the type hints and docstrings in `mcp_server.py`.

We turn them into the `tools` list that OpenRouter expects. This replaces the hardcoded `TOOLS` from sprint 3.

In [3]:
async def load_tools(mcp_client) -> list[dict]:
    result = await mcp_client.list_tools()
    return [
        {
            "type": "function",
            "function": {
                "name": tool.name,
                "description": tool.description or "",
                "parameters": tool.input_schema,
            },
        }
        for tool in result.tools
    ]


try:
    async with Client(MCP_SERVER_URL) as mcp_client:
        tools = await load_tools(mcp_client)
except Exception as exc:
    raise RuntimeError(
        f"Could not connect to {MCP_SERVER_URL}. "
        "Start the server first: uvicorn MLAppDev.sprint_3.mcp.mcp_server:app --port 8000"
    ) from exc

for tool in tools:
    print(tool["function"]["name"], tool["function"]["parameters"])

get_city {'type': 'object', 'properties': {'user_id': {'title': 'User Id', 'type': 'integer'}}, 'required': ['user_id'], 'title': 'get_cityArguments'}
get_weather {'type': 'object', 'properties': {'city': {'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'get_weatherArguments'}


# 4. The loop

Same loop as sprint 3. The only change is the tool call: `await mcp_client.call_tool(name, args)` instead of calling a local Python function.

`max_steps` is only a guard against a model that keeps asking for tools.

In [4]:
async def ask(messages: list, max_steps: int = 5) -> str:
    async with Client(MCP_SERVER_URL) as mcp_client:
        tools = await load_tools(mcp_client)

        for _ in range(max_steps):
            completion = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=tools,
                tool_choice="auto",
                temperature=0,
            )
            message = completion.choices[0].message
            messages.append(message)
            print(
                completion.choices[0].finish_reason,
                "|",
                message.content or [(c.function.name, c.function.arguments) for c in message.tool_calls],
            )

            if not message.tool_calls:
                return message.content

            for call in message.tool_calls:
                args = json.loads(call.function.arguments)
                result = await mcp_client.call_tool(call.function.name, args)
                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": call.id,
                        "content": result.content[0].text,
                    }
                )

    return "stopped: the model kept asking for tools"

# 5. The same questions

The city is in the question, so one tool is enough.

In [5]:
messages = [{"role": "user", "content": "What's the weather in Berlin?"}]
print(await ask(messages))

tool_calls | [('get_weather', '{"city":"Berlin"}')]
stop | The weather in Berlin is +11°C.
The weather in Berlin is +11°C.


No city in the question. `get_city` needs a `user_id` and its description says not to guess one, so the model asks for it and stops.

The second call reuses the same `messages`, so the id the user typed is in the conversation.

In [6]:
messages = [{"role": "user", "content": "What's the weather today?"}]
print(await ask(messages))

messages.append({"role": "user", "content": "My user id is 2"})
print(await ask(messages))

stop | I need to know your user ID to find out your location. Could you please provide it?
I need to know your user ID to find out your location. Could you please provide it?
tool_calls | [('get_city', '{"user_id":2}')]
tool_calls | [('get_weather', '{"city":"Madrid"}')]
stop | The weather today in Madrid is +17°C.
The weather today in Madrid is +17°C.


Two cities in one question. The model can ask for two `get_weather` calls in one turn.

In [7]:
messages = [{"role": "user", "content": "What's the weather today in Berlin and Madrid?"}]
print(await ask(messages))

tool_calls | [('get_weather', '{"city":"Berlin"}'), ('get_weather', '{"city":"Madrid"}')]
stop | The weather in Berlin is 11°C and in Madrid is 17°C.
The weather in Berlin is 11°C and in Madrid is 17°C.


# Recap

| Question | Tools the model called |
|---|---|
| "What's the weather in Berlin?" | `get_weather` only |
| "What's the weather today?" | none, it asked for the user_id |
| "My user id is 2" | `get_city`, then `get_weather` |

Nothing in the notebook chose a tool. The model picked from the descriptions returned by `list_tools()`, and the notebook sent each choice to the MCP server.

The tool list came from the server at runtime. If you add a third `@mcp.tool()` to `mcp_server.py` and restart it, the notebook picks it up without changing `TOOLS`.

# Optional: connect an assistant

The server is not tied to this notebook. The same URL can be used by an MCP-capable assistant or another client. For example, in a VS Code `mcp.json`:

```json
{
  "servers": {
    "city-weather": {
      "type": "http",
      "url": "http://127.0.0.1:8000/mcp"
    }
  }
}
```

The assistant sees the same two tools, because the tools come from the server.